# 05j-e — Architecture reassessment

05j-d found a qualified direct-tree signal but failed every absolute gate. This notebook does not train another candidate. It reconstructs the exact frozen checkpoints and determines whether the remaining error is dominated by optimizer variance, static segment bias, unresolved paired-future information, or localized morphology regimes. Development is diagnostic only; held-out data and rollout remain sealed.

## 1. Coherent checkout and GPU runtime

In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
if not ELM_REPO.exists(): subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'pyarrow', 'pyyaml'], check=True)
sys.path.insert(0, str(ELM_REPO))
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Revision:', REVISION)

In [ ]:
import h5py, json, numpy as np, pandas as pd, pyarrow, torch, yaml
assert torch.cuda.is_available(), 'Attiva una GPU Kaggle prima di eseguire 05j-e.'
print({'torch': torch.__version__, 'cuda': torch.cuda.get_device_name(0)})

## 2. Immutable artifacts and exact index matching

In [ ]:
import hashlib, shutil, zipfile
from src.hayflow_model.hines_state_normalization_repair import EXPECTED_05H_INDEX_SHA256
from src.hayflow_model.hines_netcon_semantic_repair import EXPECTED_05I_INDEX_SHA256
from src.hayflow_model.hines_synaptic_domain_repair import EXPECTED_05IB_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_recheck import EXPECTED_05IC_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_revision import EXPECTED_05J_INDEX_SHA256
from src.hayflow_model.hines_spatial_support_revision import EXPECTED_05JB_INDEX_SHA256
from src.hayflow_model.hines_trainable_topology_canary import EXPECTED_05JC_INDEX_SHA256
from src.hayflow_model.hines_architecture_reassessment import EXPECTED_05JD_INDEX_SHA256
INPUT_ROOT = Path('/kaggle/input')
def extract_zip_safely(source, destination):
    source, destination = Path(source), Path(destination); marker = destination / '.source_size'; stamp = str(source.stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp: return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True); root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve(); assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp); return destination
def index_matches(path, expected):
    path = Path(path)
    try:
        if path.is_file():
            with zipfile.ZipFile(path) as archive:
                names = [n for n in archive.namelist() if n.replace('\\', '/').endswith('artifact_index.json')]
                return len(names) == 1 and hashlib.sha256(archive.read(names[0])).hexdigest() == expected
        index = path / 'artifact_index.json'; return index.is_file() and hashlib.sha256(index.read_bytes()).hexdigest() == expected
    except (OSError, zipfile.BadZipFile): return False
def artifact(env, archive_name, marker, expected):
    candidates = ([Path(os.environ[env]).expanduser()] if os.environ.get(env) else []) + list(INPUT_ROOT.rglob(archive_name)) + [p.parent for p in INPUT_ROOT.rglob(marker)]
    valid = [p.resolve() for p in candidates if p.exists() and index_matches(p, expected)]
    assert valid, f'{archive_name} non trovato o con indice incompatibile. Candidati: {[str(p) for p in candidates]}'
    return valid[0]
topup_candidates = ([Path(os.environ['HAYFLOW_TOPUP_V3']).expanduser()] if os.environ.get('HAYFLOW_TOPUP_V3') else []) + list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip')) + [p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE = next((p.resolve() for p in topup_candidates if p.exists()), None); assert TOPUP_SOURCE is not None, 'Top-up BAP v3 non trovato.'
TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05je_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifests = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json')); assert len(manifests) == 1, manifests
COMPOSITE_MANIFEST = manifests[0]
base_candidates = ([Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else []) + [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()] + [p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE = next((p.resolve() for p in base_candidates if p.exists()), None); assert BASE_SOURCE is not None, 'Dataset base targeted v1.1 non trovato.'
CHECKPOINT_05B_SOURCE = next((p.resolve() for p in list(INPUT_ROOT.rglob('hayflow_hines_canary_v2.zip')) + [p.parent for p in INPUT_ROOT.rglob('canary_models.pt')] if p.exists()), None); assert CHECKPOINT_05B_SOURCE is not None, 'Artefatto 05b non trovato.'
if CHECKPOINT_05B_SOURCE.name == 'checkpoints': CHECKPOINT_05B_SOURCE = CHECKPOINT_05B_SOURCE.parent
ARTIFACT_05C_SOURCE = next((p.resolve() for p in list(INPUT_ROOT.rglob('hayflow_hines_causal_isolation.zip')) + [p.parent for p in INPUT_ROOT.rglob('checkpoint_forensics.json')] if p.exists()), None); assert ARTIFACT_05C_SOURCE is not None
ARTIFACT_05D_SOURCE = next((p.resolve() for p in list(INPUT_ROOT.rglob('hayflow_hines_residual_conditioning.zip')) + [p.parent for p in INPUT_ROOT.rglob('free_residual_report.json')] if p.exists()), None); assert ARTIFACT_05D_SOURCE is not None
ARTIFACT_05E_SOURCE = next((p.resolve() for p in list(INPUT_ROOT.rglob('hayflow_hines_segment_capacity.zip')) + [p.parent for p in INPUT_ROOT.rglob('capacity_probe_report.json')] if p.exists()), None); assert ARTIFACT_05E_SOURCE is not None
ARTIFACT_05F_SOURCE = next((p.resolve() for p in list(INPUT_ROOT.rglob('hayflow_hines_segment_micro_canary.zip')) + [p.parent for p in INPUT_ROOT.rglob('micro_canary_report.json')] if p.exists()), None); assert ARTIFACT_05F_SOURCE is not None
ARTIFACT_05G_SOURCE = next((p.resolve() for p in list(INPUT_ROOT.rglob('hayflow_hines_optimization_audit.zip')) + [p.parent for p in INPUT_ROOT.rglob('optimization_support.json')] if p.exists()), None); assert ARTIFACT_05G_SOURCE is not None
ARTIFACT_05H_SOURCE = artifact('HAYFLOW_05H_ARTIFACT', 'hayflow_hines_representation_forensics.zip', 'representation_forensics_config.json', EXPECTED_05H_INDEX_SHA256)
ARTIFACT_05I_SOURCE = artifact('HAYFLOW_05I_ARTIFACT', 'hayflow_hines_state_normalization_repair.zip', 'state_normalization_repair_config.json', EXPECTED_05I_INDEX_SHA256)
ARTIFACT_05IB_SOURCE = artifact('HAYFLOW_05IB_ARTIFACT', 'hayflow_hines_netcon_semantic_state_repair.zip', 'netcon_semantic_repair_config.json', EXPECTED_05IB_INDEX_SHA256)
ARTIFACT_05IC_SOURCE = artifact('HAYFLOW_05IC_ARTIFACT', 'hayflow_hines_synaptic_domain_repair.zip', 'synaptic_domain_repair_config.json', EXPECTED_05IC_INDEX_SHA256)
ARTIFACT_05J_SOURCE = artifact('HAYFLOW_05J_ARTIFACT', 'hayflow_hines_repaired_representation_recheck.zip', 'repaired_representation_recheck_config.json', EXPECTED_05J_INDEX_SHA256)
ARTIFACT_05JB_SOURCE = artifact('HAYFLOW_05JB_ARTIFACT', 'hayflow_hines_repaired_representation_revision.zip', 'repaired_representation_revision_config.json', EXPECTED_05JB_INDEX_SHA256)
ARTIFACT_05JC_SOURCE = artifact('HAYFLOW_05JC_ARTIFACT', 'hayflow_hines_spatial_support_revision.zip', 'spatial_support_revision_config.json', EXPECTED_05JC_INDEX_SHA256)
ARTIFACT_05JD_SOURCE = artifact('HAYFLOW_05JD_ARTIFACT', 'hayflow_hines_trainable_topology_decoder_micro_canary.zip', 'trainable_topology_canary_config.json', EXPECTED_05JD_INDEX_SHA256)
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE), '05j-d': str(ARTIFACT_05JD_SOURCE)})

## 3. Composite dataset and cryptographic preflight

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now); percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9); eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05j-e][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True); hash_last[name] = percent
bundle = prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST, base_source=BASE_SOURCE, progress=hash_progress)
display({'valid': bundle.manifest['valid'], 'fingerprint': bundle.fingerprint, 'transition_count': bundle.transition_count})
assert bundle.manifest['valid'] and bundle.transition_count == 29880 and not bundle.manifest['physical_merge_performed']

## 4. Reassessment session — no candidate training

In [ ]:
from src.hayflow_model import HinesArchitectureReassessment, HinesArchitectureReassessmentConfig, HinesCapacityConfig, HinesConditioningConfig, HinesIsolationConfig, HinesNetConSemanticRepairConfig, HinesOptimizationAuditConfig, HinesPrototypeExperimentConfig, HinesRepairedRepresentationRecheckConfig, HinesRepairedRepresentationRevisionConfig, HinesRepresentationForensicsConfig, HinesSegmentCanaryConfig, HinesSpatialSupportRevisionConfig, HinesStateNormalizationRepairConfig, HinesSynapticDomainRepairConfig, HinesTrainableTopologyCanaryConfig
base_config = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_optimization_audit.yml').read_text()); forensic_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_representation_forensics.yml').read_text()); repair_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_state_normalization_repair.yml').read_text()); netcon_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_netcon_semantic_repair.yml').read_text()); domain_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_synaptic_domain_repair.yml').read_text()); recheck_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_repaired_representation_recheck.yml').read_text()); revision_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_repaired_representation_revision.yml').read_text()); spatial_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_spatial_support_revision.yml').read_text()); topology_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_trainable_topology_canary.yml').read_text()); reassessment_payload = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_architecture_reassessment.yml').read_text())
model_config = HinesPrototypeExperimentConfig.from_mapping(base_config['model_experiment']); isolation_config = HinesIsolationConfig.from_mapping(base_config['isolation']); conditioning_config = HinesConditioningConfig.from_mapping(base_config['conditioning']); capacity_config = HinesCapacityConfig.from_mapping(base_config['capacity']); canary_config = HinesSegmentCanaryConfig.from_mapping(base_config['micro_canary']); audit_config = HinesOptimizationAuditConfig.from_mapping(base_config['optimization_audit']); representation_config = HinesRepresentationForensicsConfig.from_mapping(forensic_payload['representation_forensics']); repair_config = HinesStateNormalizationRepairConfig.from_mapping(repair_payload['state_normalization_repair']); netcon_config = HinesNetConSemanticRepairConfig.from_mapping(netcon_payload['netcon_semantic_repair']); domain_config = HinesSynapticDomainRepairConfig.from_mapping(domain_payload['synaptic_domain_repair']); recheck_config = HinesRepairedRepresentationRecheckConfig.from_mapping(recheck_payload['repaired_representation_recheck']); revision_config = HinesRepairedRepresentationRevisionConfig.from_mapping(revision_payload['repaired_representation_revision']); spatial_config = HinesSpatialSupportRevisionConfig.from_mapping(spatial_payload['spatial_support_revision']); topology_config = HinesTrainableTopologyCanaryConfig.from_mapping(topology_payload['trainable_topology_canary']); reassessment_config = HinesArchitectureReassessmentConfig.from_mapping(reassessment_payload['architecture_reassessment'])
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_architecture_reassessment')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
session = HinesArchitectureReassessment(bundle, OUTPUT_DIR, model_config, isolation_config, conditioning_config, capacity_config, canary_config, audit_config, representation_config, CHECKPOINT_05B_SOURCE, ARTIFACT_05C_SOURCE, ARTIFACT_05D_SOURCE, ARTIFACT_05E_SOURCE, ARTIFACT_05F_SOURCE, ARTIFACT_05G_SOURCE, repair_config=repair_config, artifact_05h_source=ARTIFACT_05H_SOURCE, netcon_config=netcon_config, artifact_05i_source=ARTIFACT_05I_SOURCE, domain_config=domain_config, artifact_05ib_source=ARTIFACT_05IB_SOURCE, recheck_config=recheck_config, artifact_05ic_source=ARTIFACT_05IC_SOURCE, revision_config=revision_config, artifact_05j_source=ARTIFACT_05J_SOURCE, spatial_config=spatial_config, artifact_05jb_source=ARTIFACT_05JB_SOURCE, topology_config=topology_config, artifact_05jc_source=ARTIFACT_05JC_SOURCE, reassessment_config=reassessment_config, artifact_05jd_source=ARTIFACT_05JD_SOURCE, code_revision=REVISION)
prepare_report = session.prepare_architecture_reassessment()
display({'revision': REVISION, '05j-d': prepare_report['artifact_05jd'], 'mode': prepare_report['mode']})
assert prepare_report['mode'] == 'frozen_checkpoint_forensics_no_retraining'
assert not prepare_report['development_used_for_model_selection'] and not prepare_report['heldout_inputs_extracted'] and not prepare_report['rollout_performed']

## 5. Deterministic feature reconstruction and fixed ridge reference

In [ ]:
normalizer_report = session.apply_verified_synaptic_domain_normalizer()
support_report = session.build_expanded_train_support()
feature_report = session.prepare_expanded_spatial_features()
design_report = session.prepare_topology_canary_designs()
ridge_report = session.fit_fixed_tree_ridge_baseline()
display({'design_valid': design_report['valid'], 'fit_pairs': design_report['fit_pair_count'], 'calibration_pairs': design_report['calibration_pair_count'], 'feature_width': design_report['feature_width'], 'ridge_lambda': ridge_report['selected_ridge_lambda']})
assert design_report['valid'] and design_report['normalization_fit_roles'] == ['fit']
assert not ridge_report['development_used_for_selection']

## 6. Exact frozen-checkpoint reconstruction

In [ ]:
reconstruction_report = session.reconstruct_frozen_checkpoints()
display(pd.DataFrame(reconstruction_report['runs']))
assert reconstruction_report['valid'] and not reconstruction_report['retraining_performed']
assert not reconstruction_report['development_used_for_model_selection'] and not reconstruction_report['heldout_inputs_extracted']

## 7. Error anatomy: segments, regions, seed consensus and morphology frequency

In [ ]:
anatomy_report = session.run_error_anatomy()
display({role: {'rmse_mv': row['metrics']['aggregate_voltage_rmse_mv'], 'max_error_mv': row['metrics']['maximum_segment_error_mv'], 'disagreement_ratio': row['disagreement_to_error_ratio'], 'top_segment_energy': row['top_segment_error_energy_fraction'], 'high_frequency_energy': row['high_frequency_error_energy_fraction']} for role, row in anatomy_report['roles'].items()})
display({'systematic_calibration': anatomy_report['systematic_consensus_on_calibration'], 'systematic_development': anatomy_report['systematic_consensus_on_development'], 'segment_concentrated': anatomy_report['development_segment_concentrated'], 'high_frequency': anatomy_report['development_high_frequency_dominant']})
assert anatomy_report['valid'] and not anatomy_report['retraining_performed'] and not anatomy_report['heldout_inputs_extracted']

## 8. Fit-only bias/capacity oracle and paired-future identifiability

In [ ]:
capacity_report = session.run_bias_and_capacity_controls()
identifiability_report = session.run_branch_identifiability_audit()
display({'affine_material': capacity_report['static_affine_repair_material_on_calibration_and_development'], 'rmse_gains': {role: row['rmse_improvement_fraction'] for role, row in capacity_report['roles'].items()}})
display({'collision_threshold': identifiability_report['collision_distance_threshold'], 'roles': identifiability_report['roles']})
assert capacity_report['diagnostic_oracle_only'] and not capacity_report['candidate_authorization']
assert not capacity_report['development_used_to_fit_calibrator'] and not capacity_report['calibration_used_to_fit_calibrator']
assert not identifiability_report['retraining_performed'] and not identifiability_report['heldout_inputs_extracted']

## 9. Causal diagnosis and scoped route

In [ ]:
final_report = session.finalize_architecture_reassessment(reconstruction_report, anatomy_report, capacity_report, identifiability_report)
display({'valid': final_report['valid'], 'diagnosis': final_report['diagnosis'], 'flags': final_report['diagnostic_flags'], 'next_step': final_report['next_step']})
assert final_report['valid'] and not final_report['canary_passed'] and not final_report['micro_rollout_authorized'] and not final_report['full_training_authorized']
assert not final_report['methodology']['retraining_performed']
assert not final_report['methodology']['development_used_for_model_selection']
assert not final_report['methodology']['heldout_inputs_extracted']
assert not final_report['methodology']['rollout_performed']

## 10. Create and download the diagnostic ZIP

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, display
zip_base = Path('/kaggle/working/hayflow_hines_architecture_reassessment')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
payload = base64.b64encode(zip_path.read_bytes()).decode('ascii'); filename = zip_path.name
display(Javascript(f"""
const binary = atob('{payload}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
"""))
print({'zip': str(zip_path), 'size_mib': round(zip_path.stat().st_size / 2**20, 2), 'download': 'avviato dal browser'})